# NB04 — Cross-Split Robustness

This notebook checks whether the **same modeling conclusion** survives across split strategies.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install anndata scanpy torch pandas scikit-learn scipy

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 71.0 MB/s eta 0:00:00


In [ ]:
import os, random, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import anndata as ad

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class CFG:
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    base_dir: str = "/content/drive/MyDrive/ChemDFM/canonical_q1"
    split_cols: tuple = ("split_ho_pathway","split_random","split_tyrosine_ood","split_cellcycle_ood")
    pca_dim: int = 25
    batch_size: int = 512
    epochs: int = 6
    lr: float = 1e-3
    wd: float = 1e-5
    emb_dim: int = 32
    hidden: int = 256
    dose_hidden: int = 32
    alpha_topk: float = 2.0
    alpha_mmd: float = 0.05
    control_name: str = "control"
cfg=CFG()
OUT_DIR=f"{cfg.base_dir}/results_nb04"; os.makedirs(OUT_DIR, exist_ok=True)

adata=ad.read_h5ad(cfg.data_path)
if "dose" not in adata.obs.columns and "dose_val" in adata.obs.columns: adata.obs["dose"]=adata.obs["dose_val"]
X=adata.X
if hasattr(X,"toarray"): X=X.toarray()
X=np.asarray(X,dtype=np.float32)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.1):
        super().__init__(); layers=[]; prev=in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]; prev=h
        layers.append(nn.Linear(prev,out_dim)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)
class StructuredDoseEncoder(nn.Module):
    def __init__(self, out_dim=32):
        super().__init__(); self.net=nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,out_dim))
    def forward(self,dose): return self.net(dose)
class Model(nn.Module):
    def __init__(self, latent_dim, n_drugs, n_cells):
        super().__init__()
        self.drug_emb=nn.Embedding(n_drugs, cfg.emb_dim); self.cell_emb=nn.Embedding(n_cells, cfg.emb_dim)
        self.dose_enc=StructuredDoseEncoder(cfg.dose_hidden); self.ctrl_enc=MLP(latent_dim,[cfg.hidden,cfg.hidden],cfg.hidden,0.1)
        self.delta_head=MLP(cfg.hidden+cfg.emb_dim+cfg.emb_dim+cfg.dose_hidden,[cfg.hidden,cfg.hidden],latent_dim,0.1)
    def forward(self,x0,drug_idx,cell_idx,dose):
        z=torch.cat([self.ctrl_enc(x0),self.drug_emb(drug_idx),self.cell_emb(cell_idx),self.dose_enc(dose)],dim=1)
        delta=self.delta_head(z); return delta, x0+delta

def pairwise_sq_dists(x,y):
    x_norm=(x**2).sum(dim=1,keepdim=True); y_norm=(y**2).sum(dim=1,keepdim=True).T
    return torch.clamp(x_norm+y_norm-2.0*x@y.T, min=0.0)
def rbf_mmd(x,y,gamma=None):
    if x.shape[0]<2 or y.shape[0]<2: return x.new_tensor(0.0)
    dxy=pairwise_sq_dists(x,y); dxx=pairwise_sq_dists(x,x); dyy=pairwise_sq_dists(y,y)
    if gamma is None:
        vals=dxy.flatten(); vals=vals[vals>0]
        gamma=1.0/(vals.median().item()+1e-6) if vals.numel()>0 else 1.0
    return torch.exp(-gamma*dxx).mean()+torch.exp(-gamma*dyy).mean()-2.0*torch.exp(-gamma*dxy).mean()
def cell_aware_mmd(x_hat,x_true,cell_idx,min_count=8):
    losses=[]
    for cid in torch.unique(cell_idx):
        m=(cell_idx==cid)
        if m.sum().item()>=min_count: losses.append(rbf_mmd(x_hat[m],x_true[m]))
    return torch.stack(losses).mean() if losses else x_hat.new_tensor(0.0)
def topk_mask_from_true(delta_true,k=10):
    idx=torch.topk(delta_true.abs(), k=min(k,delta_true.shape[1]), dim=1).indices
    mask=torch.zeros_like(delta_true); mask.scatter_(1, idx, 1.0); return mask
def compute_top50(pred,true,x0):
    vals=[]
    for i in range(true.shape[0]):
        idx=np.argsort(-np.abs(true[i]-x0[i]))[:50]
        vals.append(r2_score(true[i,idx], pred[i,idx]))
    return float(np.mean(vals))

rows=[]
for split_col in cfg.split_cols:
    if split_col not in adata.obs.columns: continue
    def map_split(x):
        x=str(x).lower()
        if "train" in x: return "train"
        if "test" in x or "val" in x: return "test"
        if "ood" in x: return "ood"
        return "drop"
    obs=adata.obs.copy()
    obs["_split3"]=obs[split_col].astype(str).map(map_split)
    keep=obs["_split3"].isin(["train","test","ood"]).values
    if keep.sum()==0: continue
    obs=obs[keep].copy(); Xs=X[keep]
    if obs["_split3"].eq("ood").sum()==0:
        print(f"Skip split without OOD: {split_col}")
        continue
    train_mask=(obs["_split3"]=="train").values
    pca=PCA(n_components=cfg.pca_dim, random_state=SEED)
    X_pca=np.zeros((obs.shape[0],cfg.pca_dim),dtype=np.float32)
    X_pca[train_mask]=pca.fit_transform(Xs[train_mask]).astype(np.float32)
    X_pca[~train_mask]=pca.transform(Xs[~train_mask]).astype(np.float32)
    drug_enc=LabelEncoder(); cell_enc=LabelEncoder()
    drug_enc.fit(obs["condition"].astype(str)); cell_enc.fit(obs["cell_type"].astype(str))
    obs["drug_idx"]=drug_enc.transform(obs["condition"].astype(str)); obs["cell_idx"]=cell_enc.transform(obs["cell_type"].astype(str))
    control_mask=(obs["condition"].astype(str).str.lower()==cfg.control_name).values
    ctrl_means={}
    for cell in sorted(obs["cell_type"].astype(str).unique()):
        m=control_mask & (obs["cell_type"].astype(str).values==cell)
        ctrl_means[cell]=X_pca[m].mean(axis=0).astype(np.float32)
    X0=np.stack([ctrl_means[c] for c in obs["cell_type"].astype(str).values]).astype(np.float32)
    D=(X_pca-X0).astype(np.float32)
    class DS(Dataset):
        def __init__(self, split):
            mask=(obs["_split3"].values==split) & (obs["condition"].astype(str).str.lower().values!=cfg.control_name)
            self.idxs=np.where(mask)[0]
        def __len__(self): return len(self.idxs)
        def __getitem__(self,i):
            idx=self.idxs[i]; row=obs.iloc[idx]
            return {"x_true": torch.tensor(X_pca[idx],dtype=torch.float32), "x0": torch.tensor(X0[idx],dtype=torch.float32), "delta": torch.tensor(D[idx],dtype=torch.float32), "drug_idx": torch.tensor(int(row["drug_idx"]),dtype=torch.long), "cell_idx": torch.tensor(int(row["cell_idx"]),dtype=torch.long), "dose": torch.tensor([np.log1p(max(float(row["dose"]),0.0))],dtype=torch.float32), "cell_type": str(row["cell_type"])}
    train_loader=DataLoader(DS("train"),batch_size=cfg.batch_size,shuffle=True)
    test_loader=DataLoader(DS("test"),batch_size=cfg.batch_size,shuffle=False)
    ood_loader=DataLoader(DS("ood"),batch_size=cfg.batch_size,shuffle=False)
    for variant, alpha_mmd in [("residual_only",0.0),("residual_mmd",cfg.alpha_mmd)]:
        model=Model(cfg.pca_dim, len(drug_enc.classes_), len(cell_enc.classes_)).to(DEVICE)
        opt=torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        best_score=-1e9; best=None
        for epoch in range(cfg.epochs):
            model.train()
            for b in train_loader:
                x0=b["x0"].to(DEVICE); x_true=b["x_true"].to(DEVICE); delta_true=b["delta"].to(DEVICE)
                drug_idx=b["drug_idx"].to(DEVICE); cell_idx=b["cell_idx"].to(DEVICE); dose=b["dose"].to(DEVICE)
                opt.zero_grad(); delta_hat,x_hat=model(x0,drug_idx,cell_idx,dose)
                mask=topk_mask_from_true(delta_true,10)
                loss=F.mse_loss(delta_hat,delta_true)+cfg.alpha_topk*((((delta_hat-delta_true)**2)*mask).sum()/(mask.sum()+1e-8))
                if alpha_mmd>0: loss = loss + alpha_mmd*cell_aware_mmd(x_hat,x_true,cell_idx)
                loss.backward(); opt.step()
            def eval_loader(loader):
                model.eval(); P=[]; T=[]; X0s=[]; C=[]
                with torch.no_grad():
                    for b in loader:
                        x0=b["x0"].to(DEVICE); x_true=b["x_true"].to(DEVICE)
                        drug_idx=b["drug_idx"].to(DEVICE); cell_idx=b["cell_idx"].to(DEVICE); dose=b["dose"].to(DEVICE)
                        _,x_hat=model(x0,drug_idx,cell_idx,dose)
                        P.append(x_hat.cpu().numpy()); T.append(x_true.cpu().numpy()); X0s.append(x0.cpu().numpy()); C.extend(b["cell_type"])
                P=np.concatenate(P); T=np.concatenate(T); X0s=np.concatenate(X0s); C=np.array(C)
                return compute_top50(P,T,X0s), compute_top50(P[C=="K562"], T[C=="K562"], X0s[C=="K562"])
            test_top50,k562_test=eval_loader(test_loader); ood_top50,k562_ood=eval_loader(ood_loader)
            score=0.5*test_top50+0.3*ood_top50+0.2*k562_ood
            if score>best_score: best_score=score; best={k:v.cpu().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(best)
        for split_name, loader in [("test",test_loader),("ood",ood_loader)]:
            model.eval(); P=[]; T=[]; X0s=[]; C=[]
            with torch.no_grad():
                for b in loader:
                    x0=b["x0"].to(DEVICE); x_true=b["x_true"].to(DEVICE)
                    drug_idx=b["drug_idx"].to(DEVICE); cell_idx=b["cell_idx"].to(DEVICE); dose=b["dose"].to(DEVICE)
                    _,x_hat=model(x0,drug_idx,cell_idx,dose)
                    P.append(x_hat.cpu().numpy()); T.append(x_true.cpu().numpy()); X0s.append(x0.cpu().numpy()); C.extend(b["cell_type"])
            P=np.concatenate(P); T=np.concatenate(T); X0s=np.concatenate(X0s); C=np.array(C)
            rows.append({"split_col": split_col, "split": split_name, "variant": variant, "r2_top50": compute_top50(P,T,X0s), "k562_r2_top50": compute_top50(P[C=="K562"], T[C=="K562"], X0s[C=="K562"])})
cross_split=pd.DataFrame(rows)
summary=cross_split.pivot(index=["split_col","split"], columns="variant", values=["r2_top50","k562_r2_top50"]).reset_index()
print("cross_split_summary"); print(summary)
cross_split.to_csv(f"{OUT_DIR}/cross_split_summary_long.csv", index=False)
summary.to_csv(f"{OUT_DIR}/cross_split_summary.csv", index=False)


Skip split without OOD: split_tyrosine_ood
Skip split without OOD: split_cellcycle_ood
cross_split_summary
                split_col split     r2_top50               k562_r2_top50  \
variant                         residual_mmd residual_only  residual_mmd   
0        split_ho_pathway   ood     0.557024      0.556533      0.563942   
1        split_ho_pathway  test     0.576348      0.576889      0.587872   
2            split_random   ood     0.577912      0.579024      0.587123   
3            split_random  test     0.575439      0.576702      0.583093   

                       
variant residual_only  
0            0.557709  
1            0.585496  
2            0.585307  
3            0.581360  
